In [ ]:
%matplotlib inline
import numpy as np
import sys
import xugrid
import geopandas as gpd

from gdptools import WeightGenP2P

In [ ]:
sys.path.append("../common")
from liss_settings import (
    cx,
    cx_provider,
    get_dflow_control_path,
    get_dflow_grid_name,
    get_modflow_coupling_tag,
    get_modflow_grid_name,
)

In [ ]:
domain = "gp"
resolution = "high"
boundary_condition = "chd"

In [ ]:
control_path = get_dflow_control_path(domain, resolution)
grid_name = get_dflow_grid_name(control_path)
print(grid_name)

In [ ]:
mf_grid_name = get_modflow_grid_name(
    domain=domain,
    boundary_condition=boundary_condition,
)
print(mf_grid_name)

In [ ]:
get_modflow_coupling_tag(1.0)

## Read the D-Flow FM output file

Make sure you run D-Flow FM by itself first so that there is an output NetCDF file available so that the mapping is done using the internal node order

In [ ]:
# use an output file because this is what will be available from bmi and is in the correct order
source_path = control_path.parent / "output/FlowFM_map.nc"
source_ds = xugrid.open_dataset(source_path)

In [ ]:
source_ds

In [ ]:
print(source_ds.grid.face_node_connectivity.shape)
source_ds.grid.face_node_connectivity

In [ ]:
print(source_ds.grid.node_face_connectivity.shape)
source_ds.grid.node_face_connectivity

### Convert the NetCDF data to a geodataframe

In [ ]:
source_gdf = source_ds["mesh2d_nFaces"].ugrid.to_geodataframe(name="cell")

In [ ]:
source_gdf.set_crs(32618, inplace=True)

## Open the shapefile with the location of the coastal boundaries in MODFLOW

The shapefile needs to be limited to coastal boundary locations and be in the same coordinate system as the D-Flow FM model (UTM 18N).

In [ ]:
fpth = f"../modflow/gis/{domain}/{mf_grid_name}_ghb_utm18n.shp"
print(fpth)

In [ ]:
target_coastal = gpd.read_file(
    fpth
)  # this is the shapefile with coastal boundary conditions
target_coastal

In [ ]:
target_coastal.crs

In [ ]:
ax = target_coastal.plot(alpha=0.25, column="ghb_no")
cx.add_basemap(ax, crs=target_coastal.crs, attribution=False, source=cx_provider)

## Create the D-FLOW FM to GHB mapping

In [ ]:
# generate the weights
weight_gen = WeightGenP2P(
    target_poly=target_coastal,
    target_poly_idx="ghb_no",
    source_poly=source_gdf,
    source_poly_idx=["cell"],
    method="serial",
    weight_gen_crs=32618,
)
weights = weight_gen.calculate_weights()

In [ ]:
weights[:12]

In [ ]:
map_shape = (target_coastal.shape[0], source_gdf.shape[0])
map_shape

In [ ]:
dflow2mfghb = np.zeros(map_shape, dtype=float)
print(f"{dflow2mfghb.shape}\n{dflow2mfghb}")

In [ ]:
for r, c, v in zip(weights["ghb_no"], weights["cell"], weights["wght"]):
    dflow2mfghb[int(r), int(c)] = v

## Create the ghb masking array

Where the sums of the weights along a row are not equal to ~1.0

In [ ]:
mask_idx = np.isclose(dflow2mfghb.sum(axis=1), 1.0)
print(f"{mask_idx.sum()}\n{mask_idx.shape}\n{mask_idx}")

### Test the D-FLOW FM to GHB mapping

In [ ]:
s = np.full(source_gdf.shape[0], 1.0)
h = np.full(mask_idx.shape, 2.0)
h[mask_idx] = dflow2mfghb.dot(s)[mask_idx]
s.shape, dflow2mfghb.shape, h.shape

In [ ]:
print(f"{h.sum()}\n{h}")

#### Test with a nan

In [ ]:
s = np.random.random(source_gdf.shape[0])
s[1544] = -1e30
print(s)

In [ ]:
h = np.full(mask_idx.shape, 2.0)
h = dflow2mfghb.dot(s)
h.shape

In [ ]:
print(f"{h.sum()}\n{h}")

## Create the GHB to Qext mapping

In [ ]:
ghb2qext = np.transpose(dflow2mfghb.copy())

### Test the GHB to Qext mapping

In [ ]:
q = np.full(ghb2qext.shape[1], 1.0)

In [ ]:
qext = ghb2qext.dot(q)

In [ ]:
print(f"{qext.sum()}\n{qext.shape}")

## Save the mapping arrays

In [ ]:
fpath = f"../mapping/{domain}/{resolution}/dflow_{grid_name}_to_{mf_grid_name}_ghb.npz"
np.savez_compressed(fpath, dflow2mfghb=dflow2mfghb, ghbmask=mask_idx, ghb2qext=ghb2qext)

In [ ]:
fpath